# TMCAD fine-tuning of Laya on Kaggle

This notebook uses the frozen full-study data, the same class Choice question, and the same train/calibration/validation split as the local study. It changes only execution settings for a single Kaggle GPU: FP16 and a physical batch of 8 replace FP32 and batch 2 with four accumulation steps. The effective batch remains 8. Test labels are never used for training or checkpoint/temperature selection. The test report is exploratory because earlier local results already exposed this split.

**Before Run All:** Upload the **corrected v2** `jev_tmcad_laya_finetune_v2.zip` as a **private Kaggle Dataset**, attach it under *Add Input*, choose a GPU accelerator, and enable Internet for the pinned Laya package and checkpoint download. Remove the older input Dataset version from this notebook. No STEP files, API keys, or manual path edits are required. Kaggle saves files written to `/kaggle/working` with the notebook version. The code uses one GPU even if the selected accelerator exposes two.

The input bundle contains prepared train/calibration/validation JSONL and all-sample descriptor-derived inference states. The output folder and ZIP contain epoch checkpoints, a Laya-loadable selected model, training history, per-sample inference records, direct Choice metrics/predictions/confusion tables, configs, and provenance hashes. The complete descriptor/geometry/JEV/Laya comparison remains a local `study-evaluate` step after download.


In [ ]:
from pathlib import Path
import json, sys, zipfile, subprocess

inputs = Path('/kaggle/input')
matches = [p for p in inputs.rglob('manifest.json') if p.parent.name == 'jev_tmcad_laya_ft']
if not matches:
    zips = list(inputs.rglob('jev_tmcad_laya_finetune_v2.zip'))
    if len(zips) != 1:
        raise FileNotFoundError('Attach exactly one private Dataset containing jev_tmcad_laya_finetune_v2.zip')
    unpacked = Path('/kaggle/working/unpacked_input')
    unpacked.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        if any(Path(n).is_absolute() or '..' in Path(n).parts for n in archive.namelist()):
            raise ValueError('Unsafe archive path')
        archive.extractall(unpacked)
    matches = list(unpacked.rglob('manifest.json'))
if len(matches) != 1:
    raise ValueError(f'Expected one study bundle manifest, found {len(matches)}')
bundle = matches[0].parent
print('Input bundle:', bundle)
print('Installing pinned Laya dependencies; Kaggle CUDA PyTorch is retained')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                       str(bundle / 'config/requirements_kaggle.txt')])


In [ ]:
import hashlib, importlib, platform, shutil, time
import pandas as pd
import torch
import yaml

manifest = json.loads((bundle / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['bundle_type'] == 'jev_tmcad_laya_finetune_v2', 'Attach the corrected v2 upload ZIP'
source_file = bundle / 'code/src/jev_tmcad/training/laya.py'
if '_validation_macro_f1' not in source_file.read_text(encoding='utf-8'):
    raise RuntimeError(f'The attached Dataset still has the old trainer: {source_file}. Attach the v2 ZIP.')
for relative, info in manifest['files'].items():
    path = bundle / relative
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != info['sha256']:
        raise ValueError(f'Bundle file missing or changed: {relative}')
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before training')
print('Verified', len(manifest['files']), 'files and', manifest['n_samples'], 'sample states')
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
config = yaml.safe_load((bundle / 'config/kaggle_config.yaml').read_text(encoding='utf-8'))
questions = yaml.safe_load((bundle / 'config/questions_compact.yaml').read_text(encoding='utf-8'))
sys.path.insert(0, str(bundle / 'code/src'))
# Rerunning in one Kaggle kernel can retain the old package in Python's cache.
for module_name in list(sys.modules):
    if module_name == 'jev_tmcad' or module_name.startswith('jev_tmcad.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
from jev_tmcad.training.laya import (train_laya_finetune,
    run_finetuned_laya_states, evaluate_finetuned_laya_choice, _validation_macro_f1)
import jev_tmcad.training.laya as trainer_module
if Path(trainer_module.__file__).resolve() != source_file.resolve():
    raise RuntimeError(f'Python imported {trainer_module.__file__}, expected {source_file}')
from jev_tmcad.utils import sha256_json
assert sha256_json(questions) == manifest['question_sha256']
assert f"{config['base_repository']}/{config['base_subfolder']}@{config['base_revision']}" == manifest['base_model']
assert _validation_macro_f1([{'label': 0}, {'label': 1}], [[5, 0], [0, 5]], 2) == 1.0
output = Path('/kaggle/working/jev_tmcad_laya_ft_results')
output.mkdir(parents=True, exist_ok=True)
for name in ('kaggle_config.yaml', 'questions_compact.yaml', 'local_study_full_ft.yaml',
             'requirements_kaggle.txt'):
    shutil.copy2(bundle / 'config' / name, output / name)
shutil.copy2(bundle / 'manifest.json', output / 'input_manifest.json')
shutil.copy2(bundle / 'data/metadata.json', output / 'prepared_data_metadata.json')
environment = {'python': platform.python_version(), 'platform': platform.platform(),
               'torch': torch.__version__, 'cuda': torch.version.cuda,
               'gpu': torch.cuda.get_device_name(0), 'config': config}
(output / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')


## Train and calibrate

Training uses only `train.jsonl`; the train-only calibration groups fit Choice temperature; validation macro F1 selects the epoch. Each epoch produces a weight snapshot and a history update. `laya_checkpoint.progress.json` records the current stage and batch number.

If every epoch is already complete but calibration previously failed, the next cell verifies the saved history and selected snapshot, then finishes calibration and checkpoint export. It performs no training steps. Keep the same Kaggle session and its `/kaggle/working` files. For an existing notebook session, paste `kaggle/laya/recover_from_snapshots.py` into a new cell after setup, run it, and then continue at inference.


In [ ]:
"""Kaggle notebook cell: finalize a completed Laya run without retraining.

Paste this entire file into a new cell in the *same* Kaggle session after the
notebook setup cell. It also runs automatically in the revised notebook.
"""
import gc
import json
import math
import shutil
from importlib.metadata import version

import laya
import numpy as np
import torch
from safetensors.torch import load_file

from jev_tmcad.data.annotations import file_sha256
from jev_tmcad.training.laya import _base_checkpoint, _logits, _tokenize
from jev_tmcad.utils import read_jsonl, sha256_json, write_json


def recover_completed_laya_training(bundle, output, config, questions):
    """Select the best saved epoch and perform train-only temperature fitting."""
    checkpoint = output / "laya_checkpoint"
    history_path = output / "laya_checkpoint.history.json"
    if (checkpoint / "study_metadata.json").exists():
        return json.loads((checkpoint / "study_metadata.json").read_text(encoding="utf-8"))
    if checkpoint.exists():
        raise FileExistsError(f"Partial checkpoint exists; inspect it before recovery: {checkpoint}")
    history_record = json.loads(history_path.read_text(encoding="utf-8"))
    history = history_record["epochs"]
    expected_epochs = int(config["epochs"])
    if len(history) != expected_epochs or [r["epoch"] for r in history] != list(range(1, expected_epochs + 1)):
        raise ValueError("Training history is incomplete; recovery cannot assume four finished epochs")
    for key in ("batch_size", "accumulation_steps", "precision"):
        if history_record[key] != config[key]:
            raise ValueError(f"Saved training history uses another {key}")
    scores = [float(r["validation_macro_f1"]) for r in history]
    if not all(math.isfinite(s) for s in scores):
        raise ValueError("Validation history contains a nonfinite score")
    best_index = max(range(len(scores)), key=scores.__getitem__)
    best_epoch = best_index + 1
    snapshot = output / "laya_epoch_checkpoints" / f"epoch_{best_epoch:02d}.safetensors"
    summary = json.loads(snapshot.with_suffix(".json").read_text(encoding="utf-8"))
    if (summary["epoch"] != best_epoch or
            float(summary["validation_macro_f1"]) != scores[best_index] or
            summary["model_sha256"] != file_sha256(snapshot)):
        raise ValueError("Selected epoch snapshot differs from the validation history")
    print(f"Using saved epoch {best_epoch}/{expected_epochs} (validation macro F1 {scores[best_index]:.4f}); no training batches will run.", flush=True)

    data_dir = bundle / "data"
    data_meta = json.loads((data_dir / "metadata.json").read_text(encoding="utf-8"))
    if data_meta["question_sha256"] != sha256_json(questions):
        raise ValueError("Choice question differs from the frozen fine-tuning data")
    options = list(questions["classification"]["criteria"])
    if data_meta["options"] != options:
        raise ValueError("Choice option order differs from the frozen fine-tuning data")
    data_hashes = {name: file_sha256(data_dir / f"{name}.jsonl") for name in
                   ("train", "calibration", "validation")}
    if data_hashes != data_meta["partition_hashes"]:
        raise ValueError("Frozen training, calibration, or validation data changed")
    cal_rows = read_jsonl(data_dir / "calibration.jsonl")
    if not cal_rows or any(r["split"] != "train" or r["partition"] != "calibration" for r in cal_rows):
        raise ValueError("Calibration must contain train-only samples")

    # The failed trainer may still be retained by IPython's exception traceback.
    import sys
    for name in ("last_traceback", "last_value", "last_type"):
        if hasattr(sys, name):
            setattr(sys, name, None)
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    if shell is not None and getattr(shell, "last_execution_result", None) is not None:
        shell.last_execution_result.error_in_exec = None
    gc.collect()
    torch.cuda.empty_cache()
    print("Loading pinned base Laya and fitting temperature on train-only calibration records...", flush=True)
    base, resolved = _base_checkpoint(config["base_repository"],
                                      config["base_subfolder"], config["base_revision"])
    agent = laya.load(str(base), device=config["device"])
    cal = _tokenize(cal_rows, agent.tok, agent.cfg, questions["classification"])
    agent.model.load_state_dict(load_file(str(snapshot)), strict=True)
    z = np.asarray(_logits(agent.model, cal, agent.tok.pad_token_id,
                           int(config["batch_size"]), agent.device, config["precision"]), dtype=float)
    y = np.asarray([int(item["label"]) for item in cal], dtype=int)
    if z.shape != (len(y), len(options)) or not np.isfinite(z).all() or np.any((y < 0) | (y >= len(options))):
        raise ValueError("Calibration logits or Choice indices are invalid")

    def nll(temp):
        scaled = z / temp
        scaled -= scaled.max(axis=1, keepdims=True)
        logp = scaled - np.log(np.exp(scaled).sum(axis=1, keepdims=True))
        return -float(np.mean(logp[np.arange(len(y)), y]))

    temperature = float(min(np.geomspace(.5, 5., 81), key=nll))
    checkpoint.mkdir(parents=True)
    agent.model.encoder.save_pretrained(checkpoint / "encoder", safe_serialization=True)
    agent.tok.save_pretrained(checkpoint / "tokenizer")
    shutil.copy2(snapshot, checkpoint / "model.safetensors")
    model_cfg = dict(agent.cfg)
    model_cfg.pop("temperature_by_options", None)
    model_cfg["temperature"] = [temperature, 1.0, 1.0]
    write_json(checkpoint / "rl_agent_config.json", model_cfg)
    progress_path = output / "laya_checkpoint.progress.json"
    progress = json.loads(progress_path.read_text(encoding="utf-8")) if progress_path.exists() else {}
    meta = {"base_repository": config["base_repository"], "base_subfolder": config["base_subfolder"],
            "base_revision": resolved, "laya_version": version("laya"),
            "data_metadata": str(data_dir / "metadata.json"), "data_hashes": data_hashes,
            "best_validation_macro_f1": scores[best_index], "best_epoch": best_epoch,
            "calibration_temperature": temperature, "calibration_nll": nll(temperature),
            "training_seconds": None, "elapsed_seconds_at_failure": progress.get("elapsed_seconds"),
            "accumulation_steps": int(config["accumulation_steps"]),
            "batch_size": int(config["batch_size"]), "precision": config["precision"],
            "save_epoch_snapshots": True, "epochs": history,
            "checkpoint_sha256": file_sha256(checkpoint / "model.safetensors"),
            "recovered_from_snapshot": str(snapshot),
            "recovery_note": "All training epochs completed before calibration failed; no optimizer steps were rerun."}
    write_json(checkpoint / "study_metadata.json", meta)
    del agent
    gc.collect()
    torch.cuda.empty_cache()
    return meta


if (output / "laya_checkpoint.history.json").exists() and not (output / "laya_checkpoint" / "study_metadata.json").exists():
    training = recover_completed_laya_training(bundle, output, config, questions)
    print("Recovered without retraining. Best validation epoch:", training["best_epoch"],
          "| calibration temperature:", training["calibration_temperature"])


In [ ]:
from importlib.metadata import version
environment['laya'] = version('laya')
(output / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
checkpoint = output / 'laya_checkpoint'
if not (checkpoint / 'study_metadata.json').exists():
    training = train_laya_finetune(
        bundle / 'data', checkpoint, questions,
        repository=config['base_repository'], subfolder=config['base_subfolder'],
        revision=config['base_revision'], seed=int(config['project_seed']),
        device=config['device'], epochs=int(config['epochs']),
        batch_size=int(config['batch_size']),
        accumulation_steps=int(config['accumulation_steps']),
        precision=config['precision'],
        save_epoch_snapshots=bool(config['save_epoch_snapshots']),
        encoder_lr=float(config['encoder_lr']), head_lr=float(config['head_lr']),
        weight_decay=float(config['weight_decay']), group_size=int(config['group_size']),
        sigma_start=float(config['sigma_start']), sigma_end=float(config['sigma_end']),
        rl_weight=float(config['rl_weight']))
else:
    training = json.loads((checkpoint / 'study_metadata.json').read_text(encoding='utf-8'))
pd.DataFrame(training['epochs']).to_csv(output / 'training_history.csv', index=False)
print('Best validation macro F1:', training['best_validation_macro_f1'],
      'at epoch', training['best_epoch'])


## Run the selected model on the matched cohort

The following stage uses the frozen per-sample compact states and the unchanged Choice/Noul question specification. It writes the JSONL record format expected by local `study-evaluate`. Existing complete records are reused by request hash, and the file is appended incrementally.


In [ ]:
states = bundle / 'data/inference_states.jsonl'
results = output / 'laya_results.jsonl'
if config['run_full_inference']:
    n_records = run_finetuned_laya_states(states, questions, checkpoint, results,
                                           device=config['device'])
    if n_records != manifest['n_samples']:
        raise ValueError(f'Inference coverage: {n_records} of {manifest["n_samples"]}')
    print('Fine-tuned Laya responses:', n_records)


## Direct Choice report and downloadable output

These are direct Choice metrics only. The local evaluator remains authoritative for the full method matrix and expert-graded retrieval. Test metrics are descriptive because this test split had already been inspected before fine-tuning was added.


In [ ]:
if config['run_full_inference'] and config['run_direct_choice_evaluation']:
    direct = evaluate_finetuned_laya_choice(states, results, output / 'direct_choice_evaluation')
    print(pd.DataFrame(direct['methods']).T[['n', 'accuracy', 'macro_f1', 'balanced_accuracy', 'nll', 'brier']])

readme = '''Kaggle Laya/TMCAD output\n\n'''
readme += 'laya_checkpoint/: Laya-loadable selected model (model.safetensors, encoder/, tokenizer/, rl_agent_config.json, study_metadata.json).\n'
readme += 'laya_epoch_checkpoints/: epoch weight snapshots and validation summaries.\n'
readme += 'laya_checkpoint.history.json, laya_checkpoint.progress.json, training_history.csv: training logs.\n'
readme += 'laya_results.jsonl: matched all-sample Choice/Noul predictions, request hashes, source metadata and timing.\n'
readme += 'direct_choice_evaluation/: per-sample predictions, split metrics and confusion tables.\n'
readme += 'config/*.yaml and input_manifest.json: frozen settings and source file hashes.\n\n'
readme += 'Locally, verify/import this ZIP with kaggle/laya/import_results.py, then use configs/study_full_ft_kaggle.yaml for the matched evaluator. Retrieval also requires finalized expert grades.\n'
(output / 'README_outputs.txt').write_text(readme, encoding='utf-8')
files = [p for p in output.rglob('*') if p.is_file() and p.name != 'output_manifest.json']
result_manifest = {'input_manifest_sha256': hashlib.sha256((bundle / 'manifest.json').read_bytes()).hexdigest(),
                   'model_sha256': hashlib.sha256((checkpoint / 'model.safetensors').read_bytes()).hexdigest(),
                   'files': {str(p.relative_to(output)): {'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
                                                           'bytes': p.stat().st_size} for p in sorted(files)}}
(output / 'output_manifest.json').write_text(json.dumps(result_manifest, indent=2), encoding='utf-8')
archive_path = Path('/kaggle/working/jev_tmcad_laya_ft_results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(output.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=str(Path(output.name) / path.relative_to(output)))
print('Download:', archive_path)
print('Output folder:', output)


## Optional convergence trial: epochs 5–8

The recorded validation macro F1 increased at all four original epochs (0.736, 0.810, 0.842, 0.870), so the four-epoch limit does not establish a plateau. This optional cell starts from the validated epoch-04 **model weights** and runs four more epochs using the same 6,181 training rows, 677 train-only calibration rows, 980 validation rows, FP16, physical batch 8, accumulation 1 (effective batch 8), learning rates, weight decay, and Choice CE + RL objective. The original RL noise schedule ended at 0.1; the continuation holds that endpoint. A fresh AdamW optimizer, loss scaler, and RNG state are unavoidable because the original snapshots saved weights only. This is a warm-start continuation, **not an exact uninterrupted eight-epoch run**.

Set `continuation_config['run_continuation'] = True` in the next cell after the original run is complete. A separate `continuation/` directory preserves every original result. All four new epochs are validated. The best epoch across epochs 1–8 is selected by validation macro F1. Test predictions are generated **once**, after selection, only if a new epoch beats epoch 4. Test results remain exploratory because this test set has already been inspected. Do not change accumulation alone: doing so with physical batch 8 changes the effective batch and is a different experiment.


In [ ]:
import yaml
continuation_config = yaml.safe_load('# Optional continuation after the original four-epoch run.\nrun_continuation: false\nadditional_epochs: 4\nstart_epoch: 4\nselection_metric: validation_macro_f1\ncontinuation_sigma: 0.1\nretain_original_training_settings: true\n')
# Change only this flag to True when ready to run the optional continuation.
continuation_config['run_continuation'] = False
print(continuation_config)


In [ ]:
"""Optional Kaggle notebook cell: four more epochs from saved model weights.

Paste into the *existing* Kaggle notebook after its setup cells. This is a
weight-only continuation: the original run did not save AdamW/scaler/RNG state.
No code from a replacement upload Dataset is needed.
"""
import hashlib
import json
import math
import random
import shutil
import time
from importlib.metadata import version
from pathlib import Path

import laya
import numpy as np
import torch
import torch.nn.functional as F
from laya.common import collate_items, proper_reward
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm

from jev_tmcad.data.annotations import file_sha256
from jev_tmcad.training.laya import _base_checkpoint, _logits, _tokenize, _validation_macro_f1
from jev_tmcad.utils import read_jsonl, sha256_json, write_json


def _choice_temperature(items, logits, n_classes):
    y = np.asarray([int(item["label"]) for item in items], dtype=int)
    z = np.asarray(logits, dtype=float)
    if not len(y) or z.shape != (len(y), n_classes) or not np.isfinite(z).all() or np.any((y < 0) | (y >= n_classes)):
        raise ValueError("Invalid train-only calibration logits or Choice IDs")

    def nll(temperature):
        scores = z / temperature
        scores -= scores.max(axis=1, keepdims=True)
        log_probs = scores - np.log(np.exp(scores).sum(axis=1, keepdims=True))
        return -float(np.mean(log_probs[np.arange(len(y)), y]))

    temperature = float(min(np.geomspace(.5, 5., 81), key=nll))
    return temperature, nll(temperature)


def continue_laya_from_saved_weights(bundle, output, config, questions, continuation_config):
    if not continuation_config["retain_original_training_settings"]:
        raise ValueError("This diagnostic requires the original training settings")
    if int(continuation_config["additional_epochs"]) != 4 or int(continuation_config["start_epoch"]) != 4:
        raise ValueError("This trial is defined as four epochs following the original four")
    if float(continuation_config["continuation_sigma"]) != float(config["sigma_end"]):
        raise ValueError("Keep the original RL noise schedule endpoint during continuation")
    if int(config["batch_size"]) != 8 or int(config["accumulation_steps"]) != 1 or config["precision"] != "fp16":
        raise ValueError("The continuation requires the recorded Kaggle batch and precision settings")
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle GPU")

    original_history_path = output / "laya_checkpoint.history.json"
    original = json.loads(original_history_path.read_text(encoding="utf-8"))
    prior = original["epochs"]
    if len(prior) != 4 or [row["epoch"] for row in prior] != [1, 2, 3, 4]:
        raise ValueError("Expected a complete, validated original four-epoch run")
    for key in ("batch_size", "accumulation_steps", "precision"):
        if original[key] != config[key]:
            raise ValueError(f"Original training {key} differs from continuation settings")
    prior_scores = [float(row["validation_macro_f1"]) for row in prior]
    if not all(math.isfinite(score) for score in prior_scores):
        raise ValueError("Original validation history contains a nonfinite score")
    start_epoch = 1 + max(range(4), key=prior_scores.__getitem__)
    if start_epoch != 4:
        raise ValueError("The best original epoch is not the current epoch; inspect the trajectory first")
    start_snapshot = output / "laya_epoch_checkpoints/epoch_04.safetensors"
    start_summary = json.loads(start_snapshot.with_suffix(".json").read_text(encoding="utf-8"))
    start_digest = file_sha256(start_snapshot)
    if (start_summary["model_sha256"] != start_digest or
            float(start_summary["validation_macro_f1"]) != prior_scores[3]):
        raise ValueError("The saved starting snapshot does not match the epoch-four validation record")
    original_checkpoint_meta = output / "laya_checkpoint/study_metadata.json"
    if original_checkpoint_meta.exists():
        base_meta = json.loads(original_checkpoint_meta.read_text(encoding="utf-8"))
        if base_meta["best_epoch"] != 4 or base_meta["checkpoint_sha256"] != start_digest:
            raise ValueError("The selected original checkpoint differs from the starting snapshot")

    data_dir = bundle / "data"
    data_meta = json.loads((data_dir / "metadata.json").read_text(encoding="utf-8"))
    if data_meta["question_sha256"] != sha256_json(questions):
        raise ValueError("Frozen Choice question changed")
    options = list(questions["classification"]["criteria"])
    if data_meta["options"] != options:
        raise ValueError("Frozen Choice option order changed")
    data_hashes = {name: file_sha256(data_dir / f"{name}.jsonl") for name in
                   ("train", "calibration", "validation")}
    if data_hashes != data_meta["partition_hashes"]:
        raise ValueError("Frozen fine-tuning partitions changed")
    train_rows, cal_rows, val_rows = (read_jsonl(data_dir / f"{name}.jsonl") for name in
                                     ("train", "calibration", "validation"))
    if (not train_rows or not cal_rows or not val_rows or
            any(row["split"] != "train" for row in train_rows + cal_rows) or
            any(row["split"] != "val" for row in val_rows) or
            set(row["state_hash"] for row in train_rows) & set(row["state_hash"] for row in cal_rows)):
        raise ValueError("Frozen training, calibration, and validation partitions are invalid")

    trial = output / "continuation"
    selection_path = trial / "selection.json"
    if selection_path.exists():
        return json.loads(selection_path.read_text(encoding="utf-8"))
    if trial.exists():
        raise FileExistsError(f"Partial continuation exists; inspect the saved epochs before retrying: {trial}")
    snapshot_dir = trial / "epoch_checkpoints"
    snapshot_dir.mkdir(parents=True)
    started = time.perf_counter()

    def progress(stage, **fields):
        record = {"stage": stage, "elapsed_seconds": round(time.perf_counter() - started, 1), **fields}
        temporary = trial / "progress.json.tmp"
        temporary.write_text(json.dumps(record, indent=2), encoding="utf-8")
        temporary.replace(trial / "progress.json")

    write_json(trial / "continuation_config.json", continuation_config)
    write_json(trial / "original_history.json", original)
    progress("loading", source_snapshot=str(start_snapshot))
    seed = int(config["project_seed"])
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    base, resolved = _base_checkpoint(config["base_repository"], config["base_subfolder"],
                                      config["base_revision"])
    agent = laya.load(str(base), device=config["device"])
    model, tokenizer, device = agent.model, agent.tok, agent.device
    model.load_state_dict(load_file(str(start_snapshot)), strict=True)
    if hasattr(model.encoder, "gradient_checkpointing_enable"):
        model.encoder.gradient_checkpointing_enable()
    train = _tokenize(train_rows, tokenizer, agent.cfg, questions["classification"])
    cal = _tokenize(cal_rows, tokenizer, agent.cfg, questions["classification"])
    val = _tokenize(val_rows, tokenizer, agent.cfg, questions["classification"])
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": float(config["encoder_lr"])},
        {"params": [p for name, p in model.named_parameters() if not name.startswith("encoder.")],
         "lr": float(config["head_lr"])}], weight_decay=float(config["weight_decay"]))
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    batch_size = int(config["batch_size"])
    group_size = int(config["group_size"])
    accumulation = int(config["accumulation_steps"])
    sigma = float(continuation_config["continuation_sigma"])
    rl_weight = float(config["rl_weight"])
    history = list(prior)
    best_epoch, best_score, best_snapshot = 4, prior_scores[3], start_snapshot
    print("Starting weight-only continuation from epoch 4; AdamW and GradScaler start fresh.", flush=True)
    for global_epoch in range(5, 9):
        epoch_start = time.perf_counter()
        model.train()
        order = np.random.default_rng(seed + global_epoch - 1).permutation(len(train))
        batches = range(0, len(order), batch_size)
        losses = []
        optimizer.zero_grad(set_to_none=True)
        progress("training", epoch=global_epoch, batches_done=0, batches_total=len(batches))
        for batch_number, offset in enumerate(tqdm(batches, desc=f"Continuation epoch {global_epoch}/8", leave=False), 1):
            b = collate_items([[train[int(i)]] for i in order[offset:offset + batch_size]], tokenizer.pad_token_id)
            ids, att, pos, mask, qtype, target = (b[key].to(device) for key in
                                                   ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype", "target"))
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                logits, _ = model(ids, att, pos, mask, qtype)
            logits = logits.float()
            ce = -(target * F.log_softmax(logits, -1)).sum(-1).mean()
            eps = torch.randn((group_size, *logits.shape), device=device) * sigma
            eps -= eps.mean(dim=-1, keepdim=True)
            sampled_probs = torch.softmax(logits.detach()[None, :, :] + eps, dim=-1)
            rewards = proper_reward(sampled_probs, target, qtype, mask, w_sph=.75, w_rps=1.)
            advantages = (rewards - rewards.mean(0, keepdim=True)) / rewards.std(0, keepdim=True).clamp_min(1e-6)
            sampled = torch.multinomial(sampled_probs.reshape(-1, sampled_probs.shape[-1]), 1).reshape(group_size, -1)
            logp = F.log_softmax(logits, -1).unsqueeze(0).expand(group_size, -1, -1)
            rl = -(advantages.detach() * logp.gather(-1, sampled[..., None]).squeeze(-1)).mean()
            loss = ce + rl_weight * rl
            scaler.scale(loss / accumulation).backward()
            if batch_number % accumulation == 0 or offset + batch_size >= len(order):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            losses.append(float(loss.detach().cpu()))
            if batch_number == 1 or batch_number % 25 == 0 or batch_number == len(batches):
                progress("training", epoch=global_epoch, batches_done=batch_number,
                         batches_total=len(batches), recent_loss=round(float(np.mean(losses[-25:])), 5))
        progress("validating", epoch=global_epoch)
        snapshot = snapshot_dir / f"epoch_{global_epoch:02d}.safetensors"
        save_file({key: value.detach().cpu().contiguous() for key, value in model.state_dict().items()}, snapshot)
        validation_logits = _logits(model, val, tokenizer.pad_token_id, batch_size, device, config["precision"])
        score = _validation_macro_f1(val, validation_logits, len(options))
        record = {"epoch": global_epoch, "train_loss": float(np.mean(losses)),
                  "validation_macro_f1": score, "epoch_seconds": time.perf_counter() - epoch_start}
        history.append(record)
        write_json(snapshot.with_suffix(".json"), {**record, "model_sha256": file_sha256(snapshot)})
        write_json(trial / "history.json", {"epochs": history, "best_validation_macro_f1": max(best_score, score),
                                           "batch_size": batch_size, "accumulation_steps": accumulation,
                                           "precision": config["precision"], "optimizer_state_restored": False})
        if score > best_score:
            best_epoch, best_score, best_snapshot = global_epoch, score, snapshot
        progress("epoch_complete", epoch=global_epoch, validation_macro_f1=score,
                 best_epoch=best_epoch, best_validation_macro_f1=best_score)
        print(f"Epoch {global_epoch}: validation macro F1 {score:.4f}; best epoch {best_epoch}", flush=True)

    selection = {"original_best_epoch": 4, "original_validation_macro_f1": prior_scores[3],
                 "selected_epoch": best_epoch, "selected_validation_macro_f1": best_score,
                 "validation_improvement": best_score - prior_scores[3],
                 "optimizer_state_restored": False, "loss_scaler_state_restored": False,
                 "rng_state_restored": False, "effective_batch_size": batch_size * accumulation,
                 "source_snapshot_sha256": start_digest, "data_hashes": data_hashes,
                 "question_sha256": sha256_json(questions),
                 "status": "new_checkpoint_selected" if best_epoch > 4 else "original_checkpoint_retained"}
    if best_epoch > 4:
        progress("calibrating", selected_epoch=best_epoch)
        model.load_state_dict(load_file(str(best_snapshot)), strict=True)
        calibration_logits = _logits(model, cal, tokenizer.pad_token_id, batch_size, device, config["precision"])
        temperature, calibration_nll = _choice_temperature(cal, calibration_logits, len(options))
        checkpoint = trial / "laya_checkpoint"
        checkpoint.mkdir()
        model.encoder.save_pretrained(checkpoint / "encoder", safe_serialization=True)
        tokenizer.save_pretrained(checkpoint / "tokenizer")
        shutil.copy2(best_snapshot, checkpoint / "model.safetensors")
        model_cfg = dict(agent.cfg)
        model_cfg.pop("temperature_by_options", None)
        model_cfg["temperature"] = [temperature, 1.0, 1.0]
        write_json(checkpoint / "rl_agent_config.json", model_cfg)
        meta = {"base_repository": config["base_repository"], "base_subfolder": config["base_subfolder"],
                "base_revision": resolved, "laya_version": version("laya"),
                "data_metadata": str(data_dir / "metadata.json"), "data_hashes": data_hashes,
                "best_validation_macro_f1": best_score, "best_epoch": best_epoch,
                "calibration_temperature": temperature, "calibration_nll": calibration_nll,
                "training_seconds": time.perf_counter() - started,
                "batch_size": batch_size, "accumulation_steps": accumulation,
                "precision": config["precision"], "save_epoch_snapshots": True,
                "epochs": history, "checkpoint_sha256": file_sha256(checkpoint / "model.safetensors"),
                "continuation_from_epoch_04_sha256": start_digest,
                "optimizer_state_restored": False}
        write_json(checkpoint / "study_metadata.json", meta)
        selection["checkpoint_sha256"] = meta["checkpoint_sha256"]
    write_json(selection_path, selection)
    progress("complete", selected_epoch=best_epoch, status=selection["status"])
    return selection


In [ ]:
if continuation_config['run_continuation']:
    baseline_path = output / 'direct_choice_evaluation/direct_choice_metrics.json'
    if not baseline_path.exists():
        # A fresh Kaggle session can attach the previous notebook's downloaded ZIP
        # as a second private Dataset. Restore it before touching continuation state.
        previous = list(Path('/kaggle/input').rglob('jev_tmcad_laya_ft_results.zip'))
        if len(previous) != 1:
            raise FileNotFoundError('Attach exactly one prior jev_tmcad_laya_ft_results.zip or use the original live session')
        with zipfile.ZipFile(previous[0]) as prior_archive:
            members = prior_archive.namelist()
            if not members or any(Path(name).is_absolute() or '..' in Path(name).parts or
                                  Path(name).parts[0] != output.name for name in members):
                raise ValueError('Unsafe prior result ZIP paths')
            if f'{output.name}/output_manifest.json' not in members:
                raise ValueError('Prior result ZIP lacks its output manifest')
            prior_manifest = json.loads(prior_archive.read(f'{output.name}/output_manifest.json'))
            for name, info in prior_manifest['files'].items():
                payload = prior_archive.read(f'{output.name}/{name}')
                if len(payload) != info['bytes'] or hashlib.sha256(payload).hexdigest() != info['sha256']:
                    raise ValueError(f'Prior result ZIP member changed: {name}')
            prior_archive.extractall('/kaggle/working')
        previous_data = json.loads((output / 'prepared_data_metadata.json').read_text(encoding='utf-8'))
        current_data = json.loads((bundle / 'data/metadata.json').read_text(encoding='utf-8'))
        if (previous_data['cohort_hash'] != current_data['cohort_hash'] or
                previous_data['question_sha256'] != current_data['question_sha256'] or
                previous_data['partition_hashes'] != current_data['partition_hashes']):
            raise ValueError('Prior result ZIP used another frozen cohort, question, or training partition')
        print('Restored and verified prior Kaggle result ZIP')
    if not baseline_path.exists():
        raise FileNotFoundError('Complete the original inference/direct Choice report first')
    baseline_metrics = json.loads(baseline_path.read_text(encoding='utf-8'))['methods']
    original_history = json.loads((output / 'laya_checkpoint.history.json').read_text(encoding='utf-8'))['epochs']
    if abs(baseline_metrics['val']['macro_f1'] - original_history[3]['validation_macro_f1']) > 1e-10:
        raise ValueError('Original direct Choice validation metrics differ from saved epoch 4')
    selection = continue_laya_from_saved_weights(bundle, output, config, questions, continuation_config)
    trial = output / 'continuation'
    if selection['status'] == 'new_checkpoint_selected':
        extended_results = trial / 'laya_results.jsonl'
        n_records = run_finetuned_laya_states(bundle / 'data/inference_states.jsonl', questions,
                                               trial / 'laya_checkpoint', extended_results,
                                               device=config['device'])
        if n_records != manifest['n_samples']:
            raise ValueError(f'Continuation inference coverage: {n_records} of {manifest["n_samples"]}')
        extended_metrics = evaluate_finetuned_laya_choice(bundle / 'data/inference_states.jsonl',
                        extended_results, trial / 'direct_choice_evaluation')['methods']
    else:
        extended_metrics = None  # Original checkpoint wins on validation; do not inspect a worse test model.
    comparison = {'selection': selection, 'original_direct_choice': baseline_metrics,
                  'selected_continuation_direct_choice': extended_metrics,
                  'interpretation': 'Test is exploratory and was evaluated only after validation selection.'}
    (trial / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
    with zipfile.ZipFile('/kaggle/working/jev_tmcad_laya_continuation_results.zip', 'w',
                         compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(trial.rglob('*')):
            if path.is_file():
                archive.write(path, arcname=str(Path('continuation') / path.relative_to(trial)))
    print('Validation-selected epoch:', selection['selected_epoch'])
    print('Download: /kaggle/working/jev_tmcad_laya_continuation_results.zip')
    if extended_metrics is not None:
        print(pd.DataFrame({'original': baseline_metrics['test'],
                            'selected_continuation': extended_metrics['test']}))
